In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
from sklearn.linear_model import TheilSenRegressor
from sklearn.linear_model import LinearRegression

from matplotlib import gridspec
from matplotlib.patches import Circle
import scipy.stats as stats

from scipy.stats import kstest, cramervonmises
import tensorflow as tf
import tensorflow_probability as tfp
import pykrige.kriging_tools as kt
from pykrige.ok import OrdinaryKriging
import time

from ipcc_colormap import *
from utils import *


import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Myriad Pro'
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 600

coastline = gpd.read_file('/home/mizu_home/xp53/nas/home/coastlines-split-SGregion/lines.shp')
mask = np.loadtxt('mask.txt')

ipcc_blue = (112.0/255, 160.0/255, 205.0/255, 1.0)
ipcc_orange = (196.0/255, 121.0/255, 0.0/255, 1.0)

tmp_cmap = ipcc_cmap()
tmp_cmap.read_rgb_data_from_excel()
;

2025-08-12 18:44:46.789131: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-12 18:44:46.861923: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


''

In [2]:
# DATA PREPARATION
# Load historical data
rain_obs = np.loadtxt('data/sta_monthly.csv')
rain_sim_flatten = np.loadtxt('data/wrf_monthly.csv')
rain_sim = rain_sim_flatten.reshape(rain_sim_flatten.shape[0], 120, 160)

# Station location mapping
sim_sel = np.loadtxt('data/wrf_loc.csv')
sim_idx = sim_sel[:, :2].astype(int)
sta_loc = np.genfromtxt('data/sta_lookup_new.csv', delimiter=',')[:, 2:]

# Historical WRF at station locations
wrf_sta = np.array([rain_sim[:, i, j] for (i, j) in sim_idx]).T

# Grid coordinates  
longlat = np.loadtxt('data/lonlat.txt')
lons = longlat[0, :].reshape(120, 160)
lats = longlat[1, :].reshape(120, 160)

# Configuration constants
N_STATIONS = sim_idx.shape[0]  # 14 stations
N_MONTHS = 12
N_YEARS = 40
TOTAL_MONTHS = N_MONTHS * N_YEARS  # 480
FIGURE_DIR = 'figures'

# Plotting setup
month_labels = ['(a) Jan', '(b) Feb', '(c) Mar', '(d) Apr', '(e) May', '(f) Jun', 
                '(g) Jul', '(h) Aug', '(i) Sep', '(j) Oct', '(k) Nov', '(l) Dec']

# Map visualization parameters
SINGAPORE_EXTENT = [103.58, 104.12, 1.153, 1.502]  # [lon_min, lon_max, lat_min, lat_max]
MAX_CIRCLE_RADIUS = 0.02  # Maximum circle radius in degrees
STADIA_API_KEY = "9de40773-c642-4a20-bcec-d168a244e11e"

# Load coastline and mask
coastline = gpd.read_file('/home/mizu_home/xp53/nas/home/coastlines-split-SGregion/lines.shp')
mask = np.loadtxt('mask.txt')

print(f"Loaded data: {N_STATIONS} stations, {N_YEARS} years, {N_MONTHS} months")

Loaded data: 14 stations, 40 years, 12 months


In [3]:
# Import required mapping libraries
import cartopy.io.img_tiles as cimgt
from matplotlib.patches import Circle

In [4]:
# Calibrate noise parameters for all months
print("Calibrating noise parameters for each month...")
calibrated_noise = np.zeros((N_STATIONS, N_MONTHS))

for month_idx in range(N_MONTHS):
    
    month_obs = rain_obs[month_idx::N_MONTHS, :]
    month_wrf_sta = wrf_sta[month_idx::N_MONTHS, :]
    
    gp_month = gp_interpolator(P=N_STATIONS)
    gp_month.read_rainfall(month_obs, month_wrf_sta)
    gp_month.sn_converge()
    
    calibrated_noise[:, month_idx] = gp_month.sn.copy()

# Compute annual mean noise for reference
annual_mean_noise = np.mean(calibrated_noise, axis=1)

print(f"\nNoise calibration complete!")
print(f"Annual mean noise range: {np.min(annual_mean_noise):.1f} - {np.max(annual_mean_noise):.1f} mm")
print(f"Monthly noise array shape: {calibrated_noise.shape}")  # (14 stations, 12 months)

Calibrating noise parameters for each month...

Noise calibration complete!
Annual mean noise range: 41.3 - 86.5 mm
Monthly noise array shape: (14, 12)


In [5]:
# Custom Stadia Maps class for terrain background
class StadiaStamen(cimgt.Stamen):
    def _image_url(self, tile):
         x, y, z = tile
         return f"https://tiles.stadiamaps.com/tiles/stamen_terrain/{z}/{x}/{y}.png?api_key={STADIA_API_KEY}"

def create_single_noise_map(ax, station_noise, month_label, max_noise_global, show_legend=False, basemap = False):
    """
    Create a single noise map on the given axis.
    
    Args:
        ax: Matplotlib axis with PlateCarree projection
        station_noise: Array of noise values for each station (length N_STATIONS)
        month_label: Label for the month (e.g., "(a) Jan")
        max_noise_global: Global maximum noise value across all months (for consistent scaling)
        show_legend: Whether to show legend for this subplot
    """
    # Set map extent and add terrain background
    ax.set_extent(SINGAPORE_EXTENT, crs=crs.PlateCarree())
    
    # Add terrain basemap (lighter zoom level for subplot)
    if basemap:
        tiler = StadiaStamen("terrain")  
        ax.add_image(tiler, 13)  # Lower zoom level for smaller subplots
    
    # Add coastlines and gridlines
    coastline.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.8)
    
    gl = ax.gridlines(crs=crs.PlateCarree(), draw_labels=True, linewidth=0.5, 
                     color='gray', alpha=0.3, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = False
    gl.left_labels = False
    gl.xlocator = mticker.FixedLocator([103.6, 103.8, 104.0])
    gl.ylocator = mticker.FixedLocator([1.2, 1.3, 1.4])
    
    # Calculate noise threshold for color coding
    noise_threshold = np.median(station_noise)
    
    # Plot station circles
    for station_idx, (location, noise_val) in enumerate(zip(sta_loc, station_noise)):
        # Scale radius relative to global maximum for consistency across months
        radius = (noise_val / max_noise_global) * MAX_CIRCLE_RADIUS
        
        # Color coding: blue (low noise) vs red (high noise)
        color = 'blue' if noise_val < noise_threshold else 'red'
        
        circle = Circle(xy=(location[0], location[1]), radius=radius, 
                       color=color, alpha=0.6, transform=crs.PlateCarree())
        ax.add_patch(circle)
    
    # Add month label
    if month_label is not None:
        ax.text(0.03, 0.95, month_label, transform=ax.transAxes, fontsize=9, 
                verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))
    
    # Add legend to specified subplot
    if show_legend:
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', 
                   alpha=0.6, markersize=8, label='Low noise (< median)'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                   alpha=0.6, markersize=8, label='High noise (≥ median)')
        ]
        ax.legend(handles=legend_elements, loc='lower right', fontsize=7, framealpha=0.8)

def create_monthly_noise_maps(calibrated_noise, month_labels, output_path):
    """
    Create 4x3 grid of monthly noise maps.
    
    Args:
        calibrated_noise: Array of shape (N_STATIONS, N_MONTHS) 
        month_labels: List of month labels
        output_path: Path to save the figure
    """
    fig = plt.figure(figsize=(12, 14))
    gs = gridspec.GridSpec(4, 3, height_ratios=[1,1,1,1], bottom=0.05, top=0.95, 
                          left=0.05, right=0.95, wspace=0.15, hspace=0.2)
    
    # Calculate global maximum noise for consistent scaling
    max_noise_global = np.max(calibrated_noise)
    
    # Create subplots with PlateCarree projection
    axes = []
    for i in range(4):
        for j in range(3):
            ax = plt.subplot(gs[i, j], projection=crs.PlateCarree())
            axes.append(ax)
    
    # Create individual monthly maps
    for month_idx in range(N_MONTHS):
        station_noise = calibrated_noise[:, month_idx]
        show_legend = (month_idx == 11)  # Show legend on last subplot
        
        create_single_noise_map(
            axes[month_idx], 
            station_noise, 
            month_labels[month_idx], 
            max_noise_global,
            show_legend=show_legend
        )
    
    # Add main title
    fig.suptitle('Monthly Station Noise Parameters (GP Interpolation)', 
                fontsize=14, fontweight='bold', y=0.97)
    
    # Add explanation text
    explanation = ("Circle size ∝ noise level | Blue: low noise, Red: high noise\n"
                  "Consistent spatial patterns suggest station-specific rather than temporal effects")
    fig.text(0.5, 0.02, explanation, ha='center', va='bottom', fontsize=10, 
             style='italic', wrap=True)
    
    # Save figure
    fig.savefig(output_path, dpi=600, bbox_inches='tight')
    print(f'Saved monthly noise maps: {output_path}')
    
    return fig

In [6]:
def analyze_noise_patterns(calibrated_noise):
    """
    Analyze spatial vs temporal patterns in noise parameters.
    
    Args:
        calibrated_noise: Array of shape (N_STATIONS, N_MONTHS)
        
    Returns:
        dict: Analysis results
    """
    print("Analyzing spatial vs temporal patterns in noise parameters...")
    
    # Calculate statistics
    station_means = np.mean(calibrated_noise, axis=1)  # Mean across months for each station
    station_stds = np.std(calibrated_noise, axis=1)    # Temporal variability per station
    monthly_means = np.mean(calibrated_noise, axis=0)  # Mean across stations for each month
    monthly_stds = np.std(calibrated_noise, axis=0)    # Spatial variability per month
    
    # Calculate correlation matrix between months (to check spatial pattern consistency)
    month_correlations = np.corrcoef(calibrated_noise.T)
    mean_month_correlation = np.mean(month_correlations[np.triu_indices_from(month_correlations, k=1)])
    
    # Calculate variance decomposition
    total_variance = np.var(calibrated_noise)
    between_station_variance = np.var(station_means) 
    between_month_variance = np.var(monthly_means)
    
    results = {
        'station_means': station_means,
        'station_temporal_std': station_stds,
        'monthly_means': monthly_means,
        'monthly_spatial_std': monthly_stds,
        'mean_month_correlation': mean_month_correlation,
        'total_variance': total_variance,
        'between_station_variance': between_station_variance,
        'between_month_variance': between_month_variance,
        'spatial_vs_temporal_ratio': between_station_variance / between_month_variance
    }
    
    # Print summary
    print(f"\nNoise Pattern Analysis Results:")
    print(f"Mean inter-month correlation: {mean_month_correlation:.3f}")
    print(f"Between-station variance: {between_station_variance:.2f}")
    print(f"Between-month variance: {between_month_variance:.2f}") 
    print(f"Spatial/Temporal variance ratio: {results['spatial_vs_temporal_ratio']:.2f}")
    
    if results['spatial_vs_temporal_ratio'] > 1:
        print("→ Spatial patterns dominate over temporal patterns")
        print("→ Supports assumption that noise is more location- than month-dependent")
    else:
        print("→ Temporal patterns dominate over spatial patterns")
        print("→ Suggests significant seasonal model bias effects")
        
    if mean_month_correlation > 0.7:
        print("→ High inter-month correlation suggests consistent spatial patterns")
    elif mean_month_correlation > 0.5:
        print("→ Moderate inter-month correlation suggests somewhat consistent patterns")
    else:
        print("→ Low inter-month correlation suggests highly variable spatial patterns")
    
    return results

In [9]:
def create_aggregated_noise_map(calibrated_noise, sta_loc, output_path):
    """
    Create a single map showing how frequently each station has high noise across months.
    
    Args:
        calibrated_noise: Array of shape (N_STATIONS, N_MONTHS)
        sta_loc: Station coordinates
        output_path: Path to save the figure
    """
    # Calculate frequency of being in upper half (high noise) for each station
    high_noise_frequency = np.zeros(N_STATIONS)
    
    for month_idx in range(N_MONTHS):
        monthly_noise = calibrated_noise[:, month_idx]
        monthly_median = np.median(monthly_noise)
        high_noise_mask = monthly_noise > monthly_median
        high_noise_frequency += high_noise_mask
    
    # Convert to ratio (0-1 scale)
    high_noise_ratio = high_noise_frequency / N_MONTHS
    
    # Create the map
    fig, ax = plt.subplots(1, 1, figsize=[8, 10], 
                          subplot_kw={'projection': crs.PlateCarree()})
    
    # Set map extent and add terrain background
    ax.set_extent(SINGAPORE_EXTENT, crs=crs.PlateCarree())
    
    # Add terrain basemap
    tiler = StadiaStamen("terrain")  
    ax.add_image(tiler, 13)  # Good zoom level for single map
    
    # Add coastlines and gridlines
    coastline.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1.2)
    
    gl = ax.gridlines(crs=crs.PlateCarree(), draw_labels=True, linewidth=0.8, 
                     color='gray', alpha=0.4, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlocator = mticker.FixedLocator([103.6, 103.7, 103.8, 103.9, 104.0, 104.1])
    gl.ylocator = mticker.FixedLocator([1.2, 1.3, 1.4, 1.5])
    
    # # Plot black diamond markers for station locations (like other figures)
    # ax.scatter(sta_loc[:, 0], sta_loc[:, 1], s=25, facecolors='k', marker='D', 
    #            edgecolor='white', linewidth=0.5, transform=crs.PlateCarree(), zorder=5)
    
    # Plot bars at each station location
    max_bar_height = 0.04  # Maximum bar height in degrees
    bar_width = 0.015     # Bar width in degrees
    
    for station_idx, (location, ratio) in enumerate(zip(sta_loc, high_noise_ratio)):
        # Calculate bar height
        bar_height = ratio * max_bar_height
        
        # Color coding based on ratio
        if ratio >= 0.75:        # High noise in ≥75% of months
            color = 'darkred'
            alpha = 0.9
        elif ratio >= 0.5:       # High noise in ≥50% of months
            color = 'red'
            alpha = 0.8
        elif ratio >= 0.25:      # High noise in ≥25% of months
            color = 'orange'
            alpha = 0.7
        else:                    # High noise in <25% of months
            color = 'blue'
            alpha = 0.7
        
        # Create rectangle bar positioned above the station marker
        from matplotlib.patches import Rectangle
        bar_y_offset = 0.005  # Small offset above the diamond marker
        bar = Rectangle(xy=(location[0] - bar_width/2, location[1] + bar_y_offset), 
                       width=bar_width, height=bar_height,
                       color=color, alpha=alpha, transform=crs.PlateCarree(), zorder=4)
        ax.add_patch(bar)
        
        # # Add station number label below the diamond marker
        # ax.text(location[0], location[1] - 0.012, f'{station_idx+1}', 
        #        ha='center', va='top', fontsize=8, fontweight='bold',
        #        bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.2'),
        #        transform=crs.PlateCarree(), zorder=6)
    
    # Add legend
    # from matplotlib.lines import Line2D
    # legend_elements = [
    #     Line2D([0], [0], marker='s', color='w', markerfacecolor='darkred', 
    #            alpha=0.9, markersize=10, label='High noise ≥75% of months'),
    #     Line2D([0], [0], marker='s', color='w', markerfacecolor='red', 
    #            alpha=0.8, markersize=10, label='High noise 50-74% of months'),
    #     Line2D([0], [0], marker='s', color='w', markerfacecolor='orange', 
    #            alpha=0.7, markersize=10, label='High noise 25-49% of months'),
    #     Line2D([0], [0], marker='s', color='w', markerfacecolor='blue', 
    #            alpha=0.7, markersize=10, label='High noise <25% of months'),
    #     Line2D([0], [0], marker='D', color='k', markerfacecolor='k', 
    #            markersize=6, label='Station location', linestyle='None')
    # ]
    # ax.legend(handles=legend_elements, loc='upper left', fontsize=9, 
    #          framealpha=0.9, title='Station Consistency', title_fontsize=10)
    
    # Add title and explanation
    # ax.set_title('Station Noise Consistency Across 12 Months\n' + 
    #             'Bar height = Frequency of being in upper half of monthly noise distribution',
    #             fontsize=12, fontweight='bold', pad=20)
    
    # Save figure
    fig.savefig(output_path, dpi=600, bbox_inches='tight')
    print(f'Saved aggregated noise consistency map: {output_path}')
    
    # Print station analysis
    print(f"\nStation Consistency Analysis:")
    print(f"{'Station':<8} {'High Noise %':<12} {'Category'}")
    print("-" * 35)
    
    for station_idx, ratio in enumerate(high_noise_ratio):
        percentage = ratio * 100
        if ratio >= 0.75:
            category = "Consistently High"
        elif ratio >= 0.5:
            category = "Often High"
        elif ratio >= 0.25:
            category = "Sometimes High"
        else:
            category = "Consistently Low"
        
        print(f"Stn {station_idx+1:<4} {percentage:<11.1f}% {category}")
    
    return fig, high_noise_ratio

In [10]:
# Create noise pattern analysis and visualization
print("\n" + "="*60)
print("STATION NOISE CONSISTENCY ANALYSIS")
print("="*60)

# Analyze spatial vs temporal patterns
analysis_results = analyze_noise_patterns(calibrated_noise)

# Create aggregated noise consistency map
print(f"\nCreating aggregated station consistency map...")
aggregated_output = f'{FIGURE_DIR}/station_noise_consistency.png'
fig_aggregated, high_noise_ratios = create_aggregated_noise_map(
    calibrated_noise, sta_loc, aggregated_output
)

# Create single annual mean map for comparison
print(f"\nCreating annual mean noise map for reference...")
fig_annual, ax_annual = plt.subplots(1, 1, figsize=[6, 8], 
                                   subplot_kw={'projection': crs.PlateCarree()})

max_annual_noise = np.max(annual_mean_noise)
create_single_noise_map(ax_annual, annual_mean_noise, None,  
                       max_annual_noise, show_legend=False, basemap= True)
# ax_annual.scatter(sta_loc[:,0], sta_loc[:,1], s = 15, facecolors = 'k', marker = 'D', edgecolor = 'none')

# Add title and save annual map
ax_annual.set_title('Annual Mean Station Noise Parameters', fontsize=12, fontweight='bold', pad=20)
annual_output = f'{FIGURE_DIR}/annual_mean_noise_map.png'
fig_annual.savefig(annual_output, dpi=600, bbox_inches='tight')
print(f'Saved annual mean noise map: {annual_output}')

# Summary statistics
print(f"\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Global noise range: {np.min(calibrated_noise):.1f} - {np.max(calibrated_noise):.1f} mm")
print(f"Annual mean range: {np.min(annual_mean_noise):.1f} - {np.max(annual_mean_noise):.1f} mm")
print(f"Most problematic station: {np.max(annual_mean_noise):.1f} mm")
print(f"Best performing station: {np.min(annual_mean_noise):.1f} mm")
print(f"Station performance ratio: {np.max(annual_mean_noise)/np.min(annual_mean_noise):.1f}x")

# Identify consistently problematic stations
consistently_high = np.where(high_noise_ratios >= 0.75)[0]
consistently_low = np.where(high_noise_ratios < 0.25)[0]

print(f"\nConsistently problematic stations (high noise ≥75% of months): {len(consistently_high)}")
if len(consistently_high) > 0:
    print(f"Station IDs: {[i+1 for i in consistently_high]}")
    
print(f"Consistently good stations (high noise <25% of months): {len(consistently_low)}")
if len(consistently_low) > 0:
    print(f"Station IDs: {[i+1 for i in consistently_low]}")

plt.show()


STATION NOISE CONSISTENCY ANALYSIS
Analyzing spatial vs temporal patterns in noise parameters...

Noise Pattern Analysis Results:
Mean inter-month correlation: 0.472
Between-station variance: 144.72
Between-month variance: 149.78
Spatial/Temporal variance ratio: 0.97
→ Temporal patterns dominate over spatial patterns
→ Suggests significant seasonal model bias effects
→ Low inter-month correlation suggests highly variable spatial patterns

Creating aggregated station consistency map...
Saved aggregated noise consistency map: figures/station_noise_consistency.png

Station Consistency Analysis:
Station  High Noise % Category
-----------------------------------
Stn 1    16.7       % Consistently Low
Stn 2    41.7       % Sometimes High
Stn 3    16.7       % Consistently Low
Stn 4    16.7       % Consistently Low
Stn 5    75.0       % Consistently High
Stn 6    83.3       % Consistently High
Stn 7    91.7       % Consistently High
Stn 8    33.3       % Sometimes High
Stn 9    75.0       % 